In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import xgboost as xgb


In [7]:
df = pd.read_csv("../data/pre_proccesed_data.csv")


In [8]:
df['high_chol'] = (df['cholesterol'] >= 2).astype(int)

confounders = ["age", "gender", "ap_hi", "ap_lo", "smoke", "alco", "active"]

X_conf = df[confounders]
y_conf = df["high_chol"]

scaler_conf = StandardScaler()
X_conf_scaled = scaler_conf.fit_transform(X_conf)

propensity_model = LogisticRegression()
propensity_model.fit(X_conf_scaled, y_conf)

df["propensity_score"] = propensity_model.predict_proba(X_conf_scaled)[:, 1]


In [9]:
features = ["age","gender","height","weight","ap_hi","ap_lo",
            "cholesterol","gluc","smoke","alco","active"]

X = df[features]
y = df["cardio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [10]:
xgb_full = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_full.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [11]:
y_test_pred = xgb_full.predict(X_test)

print("XGBoost (Full Features) Metrics")
print("Accuracy :", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall   :", recall_score(y_test, y_test_pred))
print("F1 Score :", f1_score(y_test, y_test_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))


XGBoost (Full Features) Metrics
Accuracy : 0.7383665367933544
Precision: 0.7562479714378448
Recall   : 0.6937620961738872
F1 Score : 0.7236586691513316

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.78      0.75      6886
           1       0.76      0.69      0.72      6717

    accuracy                           0.74     13603
   macro avg       0.74      0.74      0.74     13603
weighted avg       0.74      0.74      0.74     13603

Confusion Matrix:
 [[5384 1502]
 [2057 4660]]


In [12]:
important_features = ["age","ap_hi","ap_lo","cholesterol",
                      "gluc","smoke","alco","active","weight"]

X_imp = df[important_features]
y_imp = df["cardio"]

X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_imp, y_imp, test_size=0.2, random_state=42, stratify=y_imp
)

xgb_imp = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb_imp.fit(X_train_imp, y_train_imp)

y_test_imp_pred = xgb_imp.predict(X_test_imp)

print("\nXGBoost (Important Features) Accuracy:",
      accuracy_score(y_test_imp, y_test_imp_pred))



XGBoost (Important Features) Accuracy: 0.7376314048371683


In [13]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

class LogisticRegressionScratch:
    def __init__(self, lr=0.01, epochs=3000):
        self.lr = lr
        self.epochs = epochs

    def fit(self, X, y):
        self.w = np.zeros(X.shape[1])
        self.b = 0
        for _ in range(self.epochs):
            y_hat = sigmoid(np.dot(X, self.w) + self.b)
            dw = np.dot(X.T, (y_hat - y)) / len(y)
            db = np.sum(y_hat - y) / len(y)
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict(self, X):
        return (sigmoid(np.dot(X, self.w) + self.b) >= 0.5).astype(int)

lr_scratch = LogisticRegressionScratch()
lr_scratch.fit(X_train_scaled, y_train)
y_pred_lr = lr_scratch.predict(X_test_scaled)

print("\nLogistic Regression (Scratch) Accuracy:",
      np.mean(y_pred_lr == y_test))



Logistic Regression (Scratch) Accuracy: 0.7272660442549438


In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Test predictions
y_test_pred = xgb_full.predict(X_test)

# Metrics
accuracy  = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall    = recall_score(y_test, y_test_pred)
f1        = f1_score(y_test, y_test_pred)

print(f"Test Accuracy  : {accuracy*100:.2f}%")
print(f"Precision     : {precision:.2f}")
print(f"Recall        : {recall:.2f}")
print(f"F1-Score      : {f1:.2f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_test_pred))


Test Accuracy  : 73.84%
Precision     : 0.76
Recall        : 0.69
F1-Score      : 0.72

Classification Report:

              precision    recall  f1-score   support

           0       0.72      0.78      0.75      6886
           1       0.76      0.69      0.72      6717

    accuracy                           0.74     13603
   macro avg       0.74      0.74      0.74     13603
weighted avg       0.74      0.74      0.74     13603

Confusion Matrix:

[[5384 1502]
 [2057 4660]]


In [16]:
# Training predictions
y_train_pred = xgb_full.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy  = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy*100:.2f}%")
print(f"Testing Accuracy : {test_accuracy*100:.2f}%")


Training Accuracy: 74.47%
Testing Accuracy : 73.84%
